# gathering: anti-trans legislation

This section contains an overview of everything I did to get the data ready to analyze with `spaCy`. It begins with downloading search results from `www.congress.gov`, then web scraping to get the text of the individual bills, and finally, cleaning extraneous information from the text, like extra punctuation and blank spaces.

I'm showing you the data gathering process because it can be an invisible and challenging one, involving complicated steps like using APIs (or, if APIs fail to work, web scraping) and cleaning the resulting data. Researchers generally only share the analysis they have done *after* they got the data, which makes it hard for beginners to replicate the process. It's a subject that deserves its own workshop (hopefully by yours truly).

## anti-trans legislation

Over the past several years, there has been a dramatic increase in "anti-trans legislation," that is, legislation that limits trans peoples' rights. According to the [Trans Legislation Tracker](https://translegislation.com/), "In just one month, the U.S. doubled the number of anti-trans bills being considered across the country from the previous year." Many of these bills block access to healthcare that is gender-affirming and to public places and activities, like the "bathroom bans," the bans in sports, and the bans on expressing gender identity in school. The explosion of bills across the country not only presents a serious issue with human rights by codifying discrimination, but also reflects a worrying trend about how the general public views trans peoples' existence and rights. 

For this project, I am interested in how language is being used to exacerbate discrimination against trans people by playing on assumptions about sex and gender. Is there something in the the language of the bills themselves that can reveal specific perspectives, understandings, or definitions of sex and gender? How is language being used to so effectively play on discriminatory assumptions about sex and gender?

## web scraping
To ge my data, I got the bill numbers from `www.congress.gov`, then used those numbers to scrape the plain text from the congress.gov servers. I'll go through the whole process step-by-step.

First, I got a list of the relevant bills using the regular search function on the congress.gov website. I did a search for the term "gender," and then downloaded the results to a spreadsheet. (Side note: the ability to download results from a search is super useful, and most websites won't offer that functionality. I actually spent about an hour trying to figure out how to scrape the results before I even noticed the little "download" button that appears next to "save your search." Interestingly, this button only appears after you've begun to put some filters on the search results. I guess that's to discourage downloading all of the results at once. Either that, or its pretty wonky behavior!)

![image of congress.gov search interface](./congress_gov.jpg)

Then, I opened up a Python notebook (like the one on this page), and used the `requests` library in Python to get the text of the bills. (FYI - I tried using the congress.gov API first, which is always the right thing to do! But it doesn't offer the full text of the bill. For future reference, you can request an API key here: https://api.congress.gov/)

The code for grabbing bill text begins with loading the necessary libraries (like `requests`) and the spreadsheet with the search results from congress.gov. 

In [3]:
# importing necessary libraries
import requests # for making http (web) requests
import pandas as pd # for working with tabular (spreadsheet) data
import csv # also for working with tabular data, in csv format

# saving the search results from congress.gov 
billz = pd.read_csv('in/house_gender.csv')

# creating a "dataframe" in pandas (a python library for working
# with spreadsheet like data
df = pd.DataFrame(billz)

Then, I go through the spreadsheet to pull out the bill numbers. These numbers will be fed into a function that inserts them into the URL that will access congress.gov servers. 

In [4]:
## go through each row in numbers column of our spreadsheet
## extract the number and put into a separate list
numbers = []
for row in df['number']:
    splitted = row.split(' ')
    for item in splitted:
        if item.isnumeric():
            numbers.append(item)

# function that contains a loop to insert bill numbers
# into the URL, then to grab the content and add to a new list
def scrape_bill_text(numbers):
    billz = []
    for item in numbers:
        bill = (f'hr{item}')
        url = (f'https://www.congress.gov/118/bills/{bill}/BILLS-118{bill}ih.htm')
        response = requests.get(url)
        content = response.content
        billz.append(content)
    return billz

# call the function, passing the list of numbers as parameter
full_text = scrape_bill_text(numbers)

Next, I turned the results, which is a bytes object, into a string, so that I could manipulate it. Then I saved the results, just so I could have a copy of them. 

In [6]:
strings = []
for item in full_text:
    i = str(item)
    strings.append(i)

with open('./in/full_text.txt', 'w') as f:
    # using csv.writer method from CSV package
    for item in full_text:
        f.write(str(item))

Now, finally, I can clean the text.

First, I load it back up. Then, I run a function to take out all of the leftover html and other characters that we don't want like \n, as well as any whitespaces. Finally, I save to a new text file, called "clean". 

In [7]:
# loading up the texts that we just saved
load = open('./in/full_text.txt')
data = load.read()
load.close()

# remove all the characters in the "take out" list by writing a
# loop that replaces those characters with an empty character, ''
def clean_up(text):
    take_out = ['\n', '/n', '\\n', '_', '[', ']', '<html><body><pre>', '<html><body><pre>', '  ']
    for item in take_out:
        if item in text:
            text = text.replace(item, '')
    return text


cleaned = clean_up(data)

# save plain to a separate text file
with open('./out/full_clean.txt', 'w') as f:
    # using csv.writer method from CSV package
    f.write(cleaned)

Now the text is ready to be loaded into `spaCy`! 